In [1]:
!pip install -q streamlit langchain-community langchain-text-splitters langchain-google-genai langchain-groq langgraph faiss-cpu rank_bm25 pyngrok pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 k

In [2]:
# ── 2. Setup Ngrok Token dari Colab Secrets ──────────────────────────────────
import os
from pyngrok import ngrok
from google.colab import userdata

try:
    ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
    print("✓ ngrok token berhasil dikonfigurasi!")
except Exception as e:
    print(f"⚠️ Masalah saat konfigurasi ngrok token: {e}")

✓ ngrok token berhasil dikonfigurasi!


In [3]:
%%writefile app.py
import os
import streamlit as st
from typing import List, TypedDict, Literal
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

# ── 1. Konfigurasi Halaman Streamlit ─────────────────────────────────────────
st.set_page_config(
    page_title="FinAssist AI - Financial & Investment Assistant",
    page_icon="📈",
    layout="wide"
)

st.title("📈 FinAssist AI - Financial & Investment Assistant")
st.caption("Asisten Investasi untuk Analisis Laporan Keuangan & Konsultasi Pasar Modal")

# ── 2. Sidebar Pengaturan & API Keys ─────────────────────────────────────────
with st.sidebar:
    st.subheader("⚙️ Pengaturan System")

    default_google = ""
    default_groq = ""
    try:
        from google.colab import userdata
        default_google = userdata.get('GOOGLE_API_KEY') or ""
        default_groq = userdata.get('GROQ_API_KEY') or ""
    except Exception:
        pass

    google_api_key = st.text_input("Google AI API Key", value=default_google, type="password")
    groq_api_key = st.text_input("Groq API Key", value=default_groq, type="password")

    repo_url = st.text_input(
        "URL GitHub Repository (PDF Laporan Keuangan)",
        value="https://github.com/dimasajisaka03-boop/HackTiv_FP.git"
    )

    selected_model = st.selectbox(
        "Pilih Model LLM Engine",
        ["openai/gpt-oss-120b","llama-3.3-70b-versatile", "mixtral-8x7b-32768", "deepseek-r1-distill-llama-70b"]
    )

    reset_button = st.button("🗑️ Reset Percakapan")

    # ── METODE CLEAR CACHE ───────────────────────────────────
    st.divider()
    col_reset, col_reload = st.columns(2)

    with col_reset:
        if st.button("🗑️ Reset Chat", use_container_width=True):
            st.session_state.messages = []
            st.rerun()

    with col_reload:
        if st.button("🔄 Reload Repo", use_container_width=True):
            st.cache_resource.clear()  # <-- KODE CLEAR CACHE DITARUH DI SINI
            st.success("Cache berhasil dibersihkan! Memuat ulang repositori...")
            st.rerun()

# ── 3. Validasi API Key ──────────────────────────────────────────────────────
if not google_api_key or not groq_api_key:
    st.info("Masukkan Google AI API Key dan Groq API Key di sidebar untuk memulai.", icon="🗝️")
    st.stop()

os.environ["GOOGLE_API_KEY"] = google_api_key
os.environ["GROQ_API_KEY"] = groq_api_key

# ── 4. Vector Store & Retriever Engine (FAISS) ───────────────────────────────
@st.cache_resource(show_spinner="Membangun Vector Database FAISS...")
def init_investment_retriever(repo_url_input: str, g_api_key: str):
    repo_name = repo_url_input.split("/")[-1].replace(".git", "")

    # Clone atau update repositori GitHub
    os.system(f"rm -rf {repo_name}")
    os.system(f"git clone {repo_url_input}")

    # Load Dokumen PDF menggunakan PyPDFDirectoryLoader
    loader = PyPDFDirectoryLoader(repo_name)
    docs = loader.load()

    if not docs:
        raise ValueError("Tidak ada dokumen PDF laporan keuangan yang ditemukan di repositori.")

    # Text Splitting
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # Vektor Embedding Gemini & FAISS
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

    return vectorstore.as_retriever(search_kwargs={"k": 3})

# ── 5. Agentic Investment Pipeline (LangGraph) ──────────────────────────────
@st.cache_resource(show_spinner="Membangun Graph Investment Agent...")
def build_investment_agent(_retriever, groq_key: str, model_name: str):
    llm = ChatGroq(model=model_name, temperature=0.1)

    # State Definition
    class AgentState(TypedDict):
        question: str
        original_question: str
        context: List[Document]
        answer: str
        route: str
        retry_count: int
        is_relevant: bool

    # Pydantic Schemas untuk Router & Evaluator
    class IntentRouter(BaseModel):
        route: Literal["rag", "general"] = Field(
            description="Pilih 'rag' jika pertanyaan membutuhkan data spesifik laporan keuangan/dokumen emiten. Pilih 'general' jika pertanyaan bersifat umum (sapaan, konsep dasar investasi, rasio umum, penjelasan teoritis)."
        )

    class GradeDocuments(BaseModel):
        score: Literal["yes", "no"] = Field(
            description="Apakah dokumen terambil memuat angka atau informasi relevan untuk menjawab pertanyaan investasi ini?"
        )

    # --- Node 1: Intent Classifier Router ---
    def classify_intent_node(state: AgentState):
        question = state["question"]
        router_llm = llm.with_structured_output(IntentRouter)

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Anda adalah Intent Classifier untuk asisten investasi.\n"
                       "Tentukan apakah pertanyaan pengguna butuh mencari data di dokumen Laporan Keuangan spesifik ('rag') atau hanya pertanyaan/sapaan umum investasi ('general')."),
            ("human", "Pertanyaan: {question}")
        ])

        try:
            res = (prompt | router_llm).invoke({"question": question})
            chosen_route = res.route
        except Exception:
            chosen_route = "rag"

        return {"route": chosen_route}

    # --- Node 2: Document Retriever ---
    def retrieve_node(state: AgentState):
        q = state["question"]
        docs = _retriever.invoke(q)
        return {"context": docs}

    # --- Node 3: Document Relevance Evaluator ---
    def grade_documents_node(state: AgentState):
        docs = state["context"]
        question = state["question"]
        evaluator_llm = llm.with_structured_output(GradeDocuments)

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Evaluasi apakah konteks dokumen Laporan Keuangan ini memuat informasi/angka yang relevan untuk menjawab pertanyaan."),
            ("human", "Pertanyaan: {question}\n\nKonteks Dokumen:\n{context}")
        ])

        context_text = "\n\n".join(d.page_content for d in docs)
        try:
            res = (prompt | evaluator_llm).invoke({"question": question, "context": context_text})
            is_rel = res.score == "yes"
        except Exception:
            is_rel = True

        return {"is_relevant": is_rel}

    # --- Node 4: Query Rewriter ---
    def rewrite_query_node(state: AgentState):
        question = state["question"]
        retry = state.get("retry_count", 0) + 1

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Anda adalah analis keuangan spesialis pencarian dokumen.\n"
                       "Tulis ulang pertanyaan user agar memuat istilah teknis akuntansi/keuangan yang lebih presisi."),
            ("human", "Pertanyaan Awal: {question}\n\nSatu Pertanyaan Hasil Optimasi:")
        ])

        new_q = (prompt | llm).invoke({"question": question}).content.strip()
        return {"question": new_q, "retry_count": retry}

    # --- Node 5: Financial RAG Generator ---
    def generate_rag_node(state: AgentState):
        docs = state["context"]
        question = state["original_question"]

        formatted_context = "\n\n".join(f"[Dokumen Hal. {doc.metadata.get('page', 'N/A')}]: {doc.page_content}" for doc in docs)

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Anda adalah Asisten Analis Keuangan & Investasi Profesional.\n"
                       "Tugas Anda adalah memberikan jawaban mendalam berdasarkan Laporan Keuangan yang disediakan.\n\n"
                       "Pedoman Jawaban:\n"
                       "1. Utamakan penggunaan data angka, persentase, dan fakta dari Konteks Dokumen.\n"
                       "2. Jika melakukan perhitungan rasio keuangan (misal: DER, ROE, PER, Current Ratio), sebutkan rumus dan komponen angka yang digunakan.\n"
                       "3. Sajikan poin-poin penting secara terstruktur, objektif, dan profesional.\n"
                       "4. Jika informasi pada dokumen kurang lengkap, jelaskan bagian apa yang ditemukan dan berikan catatan profesional.\n\n"
                       "Konteks Laporan Keuangan:\n{context}"),
            ("human", "{question}")
        ])

        chain = prompt | llm
        response = chain.invoke({"context": formatted_context, "question": question})
        return {"answer": response.content}

    # --- Node 6: General Investment Knowledge Generator ---
    def generate_general_node(state: AgentState):
        question = state["original_question"]

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Anda adalah Asisten Investasi & Edukator Finansial yang ramah dan cerdas.\n"
                       "Jawablah pertanyaan umum, konsep dasar investasi, sapaan, atau teori keuangan secara akurat, lugas, dan mudah dipahami oleh investor."),
            ("human", "{question}")
        ])

        chain = prompt | llm
        response = chain.invoke({"question": question})
        return {"answer": response.content}

    # --- Routing Logic ---
    def route_intent(state: AgentState) -> Literal["retrieve", "generate_general"]:
        if state["route"] == "general":
            return "generate_general"
        return "retrieve"

    def route_evaluation(state: AgentState) -> Literal["generate_rag", "rewrite_query"]:
        if state["is_relevant"] or state.get("retry_count", 0) >= 1:
            return "generate_rag"
        return "rewrite_query"

    # --- Assembly LangGraph Workflow ---
    workflow = StateGraph(AgentState)

    workflow.add_node("classify_intent", classify_intent_node)
    workflow.add_node("retrieve", retrieve_node)
    workflow.add_node("grade_documents", grade_documents_node)
    workflow.add_node("rewrite_query", rewrite_query_node)
    workflow.add_node("generate_rag", generate_rag_node)
    workflow.add_node("generate_general", generate_general_node)

    workflow.add_edge(START, "classify_intent")
    workflow.add_conditional_edges("classify_intent", route_intent)

    workflow.add_edge("retrieve", "grade_documents")
    workflow.add_conditional_edges("grade_documents", route_evaluation)
    workflow.add_edge("rewrite_query", "retrieve")

    workflow.add_edge("generate_rag", END)
    workflow.add_edge("generate_general", END)

    return workflow.compile()

# Inisialisasi Engine & Agent
try:
    retriever_engine = init_investment_retriever(repo_url, google_api_key)
    investment_agent = build_investment_agent(retriever_engine, groq_api_key, selected_model)
except Exception as e:
    st.error(f"Gagal memuat FinAssist Agent: {e}")
    st.stop()

# ── 6. Inisialisasi Manajemen Sesi Percakapan ─────────────────────────
if "chats" not in st.session_state:
    st.session_state.chats = {"Percakapan 1": []}

if "current_chat" not in st.session_state:
    st.session_state.current_chat = "Percakapan 1"

# ── 7. UI Sidebar Tambahan: Multi-Chat Session ────────────────────────
with st.sidebar:
    st.divider()
    st.subheader("💬 Sesi Percakapan")

    # Tombol Tambah Percakapan Baru
    if st.button("➕ Percakapan Baru", use_container_width=True):
        new_chat_num = len(st.session_state.chats) + 1
        new_chat_name = f"Percakapan {new_chat_num}"
        st.session_state.chats[new_chat_name] = []
        st.session_state.current_chat = new_chat_name
        st.rerun()

    # Dropdown / Selectbox untuk Pilih Sesi Percakapan Active
    chat_names = list(st.session_state.chats.keys())
    selected_chat = st.selectbox(
        "Pilih Riwayat Chat:",
        options=chat_names,
        index=chat_names.index(st.session_state.current_chat)
    )
    st.session_state.current_chat = selected_chat

    # Tombol Hapus Percakapan Aktif
    if st.button("🗑️ Hapus Sesi Ini", use_container_width=True):
        if len(st.session_state.chats) > 1:
            del st.session_state.chats[selected_chat]
            st.session_state.current_chat = list(st.session_state.chats.keys())[0]
            st.rerun()
        else:
            st.session_state.chats[selected_chat] = []
            st.rerun()

# ── 8. Tampilkan Pesan dari Sesi yang Aktif ────────────────────────────
active_messages = st.session_state.chats[st.session_state.current_chat]

for msg in active_messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ── 9. Input & Eksekusi Agent ─────────────────────────────────────────
prompt_input = st.chat_input("Tanyakan tentang laporan keuangan emiten atau teori investasi...")

if prompt_input:
    # Simpan pertanyaan user ke sesi aktif
    st.session_state.chats[st.session_state.current_chat].append({"role": "user", "content": prompt_input})
    with st.chat_message("user"):
        st.markdown(prompt_input)

    with st.chat_message("assistant"):
        status_placeholder = st.empty()
        with status_placeholder.status("📊 FinAssist Agent sedang menganalisis...", expanded=True) as status:
            try:
                status.write("🧠 **Langkah 1:** Mengidentifikasi Intent Pertanyaan...")
                initial_state = {
                    "question": prompt_input,
                    "original_question": prompt_input,
                    "retry_count": 0,
                    "context": [],
                    "answer": "",
                    "route": "",
                    "is_relevant": False
                }

                output = investment_agent.invoke(initial_state)

                if output["route"] == "general":
                    status.write("💡 **Langkah 2:** Menjawab menggunakan Basis Knowledge Investasi Umum...")
                else:
                    status.write("🔍 **Langkah 2:** Menjalankan Search FAISS Laporan Keuangan...")
                    if output.get("retry_count", 0) > 0:
                        status.write(f"🔄 **Query Optimization:** Query disempurnakan menjadi: *\"{output['question']}\"*")
                    status.write("📈 **Langkah 3:** Menganalisis data keuangan & menyusun laporan...")

                status.update(label="✅ Analisis Selesai!", state="complete", expanded=False)
                answer = output["answer"]

                if output.get("context"):
                    with st.expander("📚 Lihat Potongan Dokumen Laporan Keuangan Terkait"):
                        for idx, doc in enumerate(output["context"]):
                            st.markdown(f"**Sumber #{idx+1}** (Halaman: {doc.metadata.get('page', 'N/A')}):")
                            st.caption(doc.page_content)
                            st.divider()

            except Exception as e:
                status.update(label="❌ Terjadi Kesalahan!", state="error", expanded=False)
                answer = f"Terjadi kesalahan saat memproses data: {e}"

        st.markdown(answer)

    # Simpan jawaban asisten ke sesi aktif
    st.session_state.chats[st.session_state.current_chat].append({"role": "assistant", "content": answer})

Writing app.py


In [5]:
import time
import subprocess
from pyngrok import ngrok

# 1. Hentikan semua proses ngrok yang sedang berjalan di sistem
!pkill -9 ngrok

# 2. Matikan instance pyngrok
try:
    ngrok.kill()
except Exception:
    pass

# Beri jeda 2 detik agar port & domain benar-benar terlepas di server ngrok
time.sleep(2)

# 3. Hubungkan ulang tunnel ke port Streamlit (8501)
try:
    public_url = ngrok.connect(8501)
    print("=" * 60)
    print(f"🚀 Link Aplikasi FinAssist AI Kamu: {public_url}")
    print("=" * 60)
except Exception as e:
    print(f"⚠️ Masalah saat membuka tunnel: {e}")

# 4. Jalankan Streamlit di background
process = subprocess.Popen(["streamlit", "run", "app.py"])

🚀 Link Aplikasi FinAssist AI Kamu: NgrokTunnel: "https://lumpiness-curse-lemon.ngrok-free.dev" -> "http://localhost:8501"
